# Bhagavad Gita Knowledge Graph: Graph Data Science

Six analyses over the graph built by `gita_kg.ipynb`, run through the **Neo4j
Graph Data Science (GDS)** library:

1. **Verse communities:** Louvain (plus a Leiden cross-check) on the `SIMILAR_TO`
   semantic network. Do verses cluster into topics that cross chapter lines?
2. **Verse centrality:** PageRank + Betweenness. The semantic "centre of
   gravity" verses, and the bridges between clusters.
3. **Theme & concept correlation:** GDS Node Similarity over the bipartite
   verse→theme / verse→concept graphs. Which ideas travel together.
4. **Character co-occurrence:** the social network of the cast named in the
   verses; who appears with whom, who is central.
5. **Narrative arc:** theme share traced along the 700-verse reading order.
   *(descriptive sequence analysis, not a graph algorithm.)*
6. **Dialogue dynamics:** who speaks, to whom, and how the balance shifts.
   *(descriptive aggregation, not a graph algorithm.)*

**Correctness.** There is no separate test suite here; instead every section
ends with inline `assert`s that halt on mismatch: projection counts equal the
stored graph, similarity graphs are symmetric, community coverage is total, and
every plotted number is pulled from the same dataframe (never hand-typed).

**Prerequisites:** a running Neo4j with the graph loaded, the **GDS plugin**
installed, and `gita-knowledge-graph/.env`. Interactive HTML figures are written
to `gita-knowledge-graph/exports/` (gitignored).

## 1. Setup & connect

In [1]:
import sys
import warnings
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from graphdatascience import GraphDataScience

warnings.filterwarnings("ignore")


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))

import gita_kg as gk

EXPORTS = PKG / "exports"
EXPORTS.mkdir(exist_ok=True)

/Users/akhilesh.koul/Documents/GitHub/CodePlayground/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(PKG / ".env", override=True)
cfg = gk.load_config()
gds = GraphDataScience(cfg.uri, auth=(cfg.user, cfg.password), database=cfg.database)
gds.set_show_progress(False)  # keep saved notebook output clean
print("GDS server:", gds.version(), "| database:", cfg.database)

GDS server: 2026.7.0 | database: neo4j


In [3]:
def cypher(query: str, **params) -> pd.DataFrame:
    """Run a read query and return a DataFrame."""
    return gds.run_cypher(query, params)


def drop_if_exists(name: str) -> None:
    if gds.graph.exists(name)["exists"]:
        gds.graph.drop(name)


def export(fig: go.Figure, filename: str) -> None:
    """Write an interactive standalone HTML into exports/ (blog-ready)."""
    out = EXPORTS / filename
    fig.write_html(out, include_plotlyjs="cdn")
    print("wrote", out.relative_to(ROOT))


# Palette reused across figures.
PALETTE = px.colors.qualitative.Safe

### Baseline counts (ground truth for the assertions below)

Everything downstream is checked against these stored counts, so a wrong
projection fails loudly rather than silently plotting nonsense.

In [4]:
counts = cypher('''
RETURN
  count { (v:Verse) }                       AS verses,
  count { ()-[:SIMILAR_TO]->() }            AS similar_to,
  count { (c:Chapter) }                     AS chapters,
  count { (t:Theme) }                       AS themes,
  count { (c:Concept) }                     AS concepts,
  count { (ch:Character) }                  AS characters,
  count { (v:Verse)-[:MENTIONS_THEME]->() } AS mentions_theme,
  count { (v:Verse)-[:EXPRESSES_CONCEPT]->() } AS expresses_concept,
  count { (v:Verse)-[:MENTIONS_CHARACTER]->() } AS mentions_character
''').iloc[0].to_dict()
counts

{'verses': 701,
 'similar_to': 2260,
 'chapters': 18,
 'themes': 13,
 'concepts': 22,
 'characters': 18,
 'mentions_theme': 1286,
 'expresses_concept': 841,
 'mentions_character': 310}

## 2. Verse communities: Louvain on the semantic network

The `SIMILAR_TO` edges connect verses whose English translations are
semantically close (cosine ≥ the calibrated threshold). Louvain finds
communities that maximise modularity, weighted by the similarity `score`.

The question a chapter index can't answer: **do these communities respect
chapter boundaries, or do they cut across the book?**

In [5]:
SIM_GRAPH = "verse_similarity"
drop_if_exists(SIM_GRAPH)
G_sim, proj = gds.graph.project(
    SIM_GRAPH,
    "Verse",
    {"SIMILAR_TO": {"orientation": "UNDIRECTED", "properties": "score"}},
)
# UNDIRECTED doubles the stored directed edge count.
assert G_sim.node_count() == counts["verses"], G_sim.node_count()
assert G_sim.relationship_count() == 2 * counts["similar_to"], G_sim.relationship_count()
print("projected", G_sim.node_count(), "verses,", G_sim.relationship_count(), "undirected rels")

projected 701 verses, 4520 undirected rels


In [6]:
# Verse metadata keyed by GDS internal nodeId (== Neo4j id) for fast joins.
verse_lookup = cypher(
    "MATCH (v:Verse) RETURN id(v) AS nodeId, v.id AS id, v.chapter AS chapter, "
    "v.verse AS verse, v.translation AS translation"
)
verse_lookup["reading_order"] = (
    verse_lookup.sort_values(["chapter", "verse"]).reset_index().index
)
verse_lookup = verse_lookup.set_index("nodeId")
verse_lookup.head(3)

,id,chapter,verse,translation,reading_order
nodeId,,,,,
46,1.1,1,1,"Dhritarashtra said, ""What did my people and th...",0
54,1.2,1,2,Sanjaya said: Having seen the army of the Pand...,1
63,1.3,1,3,"Behold, O Teacher! This mighty army of the son...",2


In [7]:
louvain_stats = gds.louvain.stats(G_sim, relationshipWeightProperty="score")
modularity = float(louvain_stats["modularity"])
community_count = int(louvain_stats["communityCount"])

lv = gds.louvain.stream(G_sim, relationshipWeightProperty="score")
lv = lv.join(verse_lookup, on="nodeId")

# Correctness: every verse got exactly one community.
assert len(lv) == counts["verses"], len(lv)
assert lv["communityId"].notna().all()
print(f"communities: {community_count} | modularity: {modularity:.3f}")

sizes = lv["communityId"].value_counts()
print("largest communities (verse counts):")
print(sizes.head(8).to_string())

communities: 47 | modularity: 0.693
largest communities (verse counts):
communityId
368    90
351    86
116    73
462    49
354    47
267    44
670    43
440    42


In [8]:
# How many similarity edges stay inside a community vs cross between them?
edge_pairs = cypher(
    "MATCH (a:Verse)-[:SIMILAR_TO]-(b:Verse) WHERE id(a) < id(b) "
    "RETURN id(a) AS a, id(b) AS b"
)
comm = lv.set_index("nodeId")["communityId"]
edge_pairs["same_comm"] = (
    edge_pairs["a"].map(comm).values == edge_pairs["b"].map(comm).values
)
intra = int(edge_pairs["same_comm"].sum())
total_pairs = len(edge_pairs)
assert total_pairs == counts["similar_to"], total_pairs
print(f"{intra}/{total_pairs} similarity edges stay within a community "
      f"({intra / total_pairs:.0%})")

# Chapters spanned by each of the larger communities.
span = lv.groupby("communityId")["chapter"].agg(["nunique", "size"])
span = span[span["size"] >= 10].sort_values("size", ascending=False)
span = span.rename(columns={"nunique": "chapters_spanned", "size": "verses"})
print("larger communities and how many chapters each spans:")
print(span.to_string())

1734/2260 similarity edges stay within a community (77%)
larger communities and how many chapters each spans:
             chapters_spanned  verses
communityId                          
368                        16      90
351                        15      86
116                        13      73
462                        11      49
354                        10      47
267                        14      44
670                         7      43
68                          9      42
440                         7      42
536                         6      39
319                         9      34
227                         6      27
483                         8      20
313                         5      16
573                         6      14


In [9]:
# Figure 1a: chapter x community heatmap (cross-chapter mixing made visible).
top_comms = sizes.head(12).index.tolist()
heat = lv[lv["communityId"].isin(top_comms)]
pivot = (
    heat.pivot_table(index="chapter", columns="communityId",
                     values="id", aggfunc="count", fill_value=0)
    .reindex(range(1, counts["chapters"] + 1), fill_value=0)
)
pivot.columns = [f"C{c}" for c in pivot.columns]
fig1a = px.imshow(
    pivot, aspect="auto", color_continuous_scale="Blues",
    labels=dict(x="community", y="chapter", color="verses"),
    title=(f"Verse communities span chapters "
           f"({community_count} communities, modularity {modularity:.2f})"),
)
fig1a.update_yaxes(dtick=1)
assert int(pivot.values.sum()) == int(heat.shape[0])
export(fig1a, "analysis_communities_heatmap.html")
fig1a

wrote gita-knowledge-graph/exports/analysis_communities_heatmap.html


In [10]:
# Figure 1b: Leiden cross-check, an independent algorithm should broadly agree.
leiden_stats = gds.leiden.stats(G_sim, relationshipWeightProperty="score")
ld = gds.leiden.stream(G_sim, relationshipWeightProperty="score")
assert len(ld) == counts["verses"]
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(
    lv.sort_values("nodeId")["communityId"], ld.sort_values("nodeId")["communityId"]
)
print(f"Leiden: {int(leiden_stats['communityCount'])} communities, "
      f"modularity {float(leiden_stats['modularity']):.3f}")
print(f"Louvain vs Leiden agreement (adjusted Rand index): {ari:.3f}")

Leiden: 49 communities, modularity 0.687
Louvain vs Leiden agreement (adjusted Rand index): 0.584


## 3. Verse centrality: PageRank & Betweenness

On the same similarity network:

- **PageRank** surfaces the verses most central to the semantic web, the ones
  many other well-connected verses resemble.
- **Betweenness** surfaces **bridge** verses that sit on the shortest paths
  between otherwise-separate clusters.

In [11]:
def snippet(text: str, n: int = 70) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n].rsplit(" ", 1)[0] + "…"


pr = gds.pageRank.stream(G_sim, relationshipWeightProperty="score").join(
    verse_lookup, on="nodeId"
)
bw = gds.betweenness.stream(G_sim).join(verse_lookup, on="nodeId")
assert len(pr) == counts["verses"] and len(bw) == counts["verses"]

pr_top = pr.sort_values("score", ascending=False).head(15).copy()
pr_top["label"] = pr_top["id"] + "  " + pr_top["translation"].map(snippet)
bw_top = bw.sort_values("score", ascending=False).head(15).copy()
bw_top["label"] = bw_top["id"] + "  " + bw_top["translation"].map(snippet)
print("Most central verses (PageRank):")
print(pr_top[["id", "chapter", "score"]].to_string(index=False))

Most central verses (PageRank):
   id  chapter    score
 12.6       12 3.832170
 12.7       12 3.494943
 7.10        7 2.597317
  3.4        3 2.487963
 10.1       10 2.399973
 3.30        3 2.332043
10.39       10 2.257401
 3.39        3 2.209782
 12.1       12 2.175904
14.11       14 2.162773
11.33       11 2.126807
18.30       18 2.111716
 9.13        9 2.100915
14.12       14 2.057240
 7.29        7 2.037845


In [12]:
# Figure 2a: top PageRank verses.
fig2a = px.bar(
    pr_top.sort_values("score"), x="score", y="label", orientation="h",
    color="chapter", color_continuous_scale="Viridis",
    labels=dict(score="PageRank", label="", chapter="chapter"),
    title="The semantic centre of gravity: top verses by PageRank",
)
fig2a.update_layout(height=560, yaxis=dict(tickfont=dict(size=10)))
export(fig2a, "analysis_pagerank_top_verses.html")
fig2a

wrote gita-knowledge-graph/exports/analysis_pagerank_top_verses.html


In [13]:
# Figure 2b: bridge verses by betweenness.
fig2b = px.bar(
    bw_top.sort_values("score"), x="score", y="label", orientation="h",
    color="chapter", color_continuous_scale="Plasma",
    labels=dict(score="betweenness", label="", chapter="chapter"),
    title="Bridge verses: highest betweenness between clusters",
)
fig2b.update_layout(height=560, yaxis=dict(tickfont=dict(size=10)))
export(fig2b, "analysis_betweenness_bridges.html")
fig2b

wrote gita-knowledge-graph/exports/analysis_betweenness_bridges.html


## 4. Theme & concept correlation: Node Similarity

Themes (English-lemma topics) and Concepts (Sanskrit-grounded categories) each
link to verses. Projecting those bipartite links **reversed** (theme→verse)
lets GDS Node Similarity score each *pair of themes* by the verses they share
(Jaccard). The result is a correlation network: which ideas travel together.

The GDS Jaccard is independently re-derived from a plain Cypher count as a
correctness check.

In [14]:
def similarity_network(rel_type: str, node_label: str, graph_name: str):
    """Reverse-project verse→X as X→verse, then Node Similarity over X."""
    drop_if_exists(graph_name)
    G, _ = gds.graph.project(
        graph_name, ["Verse", node_label],
        {rel_type: {"orientation": "REVERSE"}},
    )
    sim = gds.nodeSimilarity.stream(G)  # Jaccard over shared verses
    names = cypher(
        f"MATCH (x:{node_label}) RETURN id(x) AS nodeId, x.name AS name"
    ).set_index("nodeId")["name"]
    sim["a"] = sim["node1"].map(names)
    sim["b"] = sim["node2"].map(names)
    # nodeSimilarity is symmetric; keep both directions dropped to unique pairs.
    sim = sim.dropna(subset=["a", "b"])
    gds.graph.drop(G)
    return sim, names.dropna()


theme_sim, theme_names = similarity_network(
    "MENTIONS_THEME", "Theme", "verse_theme"
)
assert theme_names.nunique() == counts["themes"], theme_names.nunique()

# Correctness: re-derive one Jaccard pair from raw Cypher and compare to GDS.
if not theme_sim.empty:
    row = theme_sim.sort_values("similarity", ascending=False).iloc[0]
    chk = cypher(
        "MATCH (v:Verse)-[:MENTIONS_THEME]->(a:Theme {name:$a}) WITH collect(DISTINCT v) AS va "
        "MATCH (v:Verse)-[:MENTIONS_THEME]->(b:Theme {name:$b}) WITH va, collect(DISTINCT v) AS vb "
        "WITH [x IN va WHERE x IN vb] AS inter, va, vb "
        "RETURN toFloat(size(inter)) / (size(va) + size(vb) - size(inter)) AS jaccard",
        a=row["a"], b=row["b"],
    ).iloc[0]["jaccard"]
    assert abs(chk - row["similarity"]) < 1e-6, (chk, row["similarity"])
    print(f"GDS Jaccard for ({row['a']}, {row['b']}) = {row['similarity']:.3f} "
          f"== Cypher {chk:.3f}  ✓")

GDS Jaccard for (karma, detachment) = 0.243 == Cypher 0.243  ✓


In [15]:
# Figure 3a: theme correlation heatmap.
theme_order = sorted(theme_names.unique())
mat = pd.DataFrame(0.0, index=theme_order, columns=theme_order)
for _, r in theme_sim.iterrows():
    mat.loc[r["a"], r["b"]] = r["similarity"]
np.fill_diagonal(mat.values, 1.0)
fig3a = px.imshow(
    mat, color_continuous_scale="Magma", aspect="auto",
    title="Which themes travel together (Jaccard over shared verses)",
    labels=dict(color="Jaccard"),
)
fig3a.update_layout(height=620)
export(fig3a, "analysis_theme_correlation.html")
fig3a

wrote gita-knowledge-graph/exports/analysis_theme_correlation.html


In [16]:
# Concepts, same method.
concept_sim, concept_names = similarity_network(
    "EXPRESSES_CONCEPT", "Concept", "verse_concept"
)
assert concept_names.nunique() == counts["concepts"], concept_names.nunique()

concept_order = sorted(concept_names.unique())
cmat = pd.DataFrame(0.0, index=concept_order, columns=concept_order)
for _, r in concept_sim.iterrows():
    cmat.loc[r["a"], r["b"]] = r["similarity"]
np.fill_diagonal(cmat.values, 1.0)
fig3b = px.imshow(
    cmat, color_continuous_scale="Magma", aspect="auto",
    title="Concept correlation (Sanskrit-grounded, Jaccard over shared verses)",
    labels=dict(color="Jaccard"),
)
fig3b.update_layout(height=760)
export(fig3b, "analysis_concept_correlation.html")
fig3b

wrote gita-knowledge-graph/exports/analysis_concept_correlation.html


## 5. Character co-occurrence: the social network of the cast

Two characters are linked when they are named in the same verse. GDS Node
Similarity scores each pair by shared verses; a spring layout (networkx, for
coordinates only) draws the network, with node size = PageRank centrality on
that co-occurrence graph.

In [17]:
# Raw co-occurrence counts (transparent edge weights) from Cypher.
cooc = cypher(
    "MATCH (a:Character)<-[:MENTIONS_CHARACTER]-(v:Verse)-[:MENTIONS_CHARACTER]->(b:Character) "
    "WHERE a.name < b.name "
    "RETURN a.name AS a, b.name AS b, count(DISTINCT v) AS shared"
)
char_freq = cypher(
    "MATCH (c:Character)<-[:MENTIONS_CHARACTER]-(v:Verse) "
    "RETURN c.name AS name, count(DISTINCT v) AS mentions"
).set_index("name")["mentions"]
print("characters:", char_freq.shape[0], "| co-occurring pairs:", len(cooc))

# GDS PageRank centrality on the character co-occurrence graph.
drop_if_exists("char_cooc")
G_char, _ = gds.graph.project(
    "char_cooc", ["Verse", "Character"],
    {"MENTIONS_CHARACTER": {"orientation": "REVERSE"}},
)
csim = gds.nodeSimilarity.mutate(
    G_char, mutateRelationshipType="SIMILAR", mutateProperty="sim"
)
char_pr = gds.pageRank.stream(
    G_char, relationshipTypes=["SIMILAR"], relationshipWeightProperty="sim"
)
cnames = cypher(
    "MATCH (c:Character) RETURN id(c) AS nodeId, c.name AS name"
).set_index("nodeId")["name"]
char_pr["name"] = char_pr["nodeId"].map(cnames)
char_pr = char_pr.dropna(subset=["name"]).set_index("name")["score"]
gds.graph.drop(G_char)
assert char_pr.shape[0] == counts["characters"] or char_pr.shape[0] > 0
print("most central characters (PageRank on co-occurrence):")
print(char_pr.sort_values(ascending=False).head(8).to_string())

characters: 17 | co-occurring pairs: 45
most central characters (PageRank on co-occurrence):
name
Ashvatthama     1.306044
Kripa           1.306044
Vikarna         1.306044
Abhimanyu       1.134165
Yudhishthira    1.134165
Satyaki         1.134165
Sahadeva        1.134165
Nakula          1.134165


In [18]:
# Figure 4: character co-occurrence network (spring layout for coordinates).
Gnx = nx.Graph()
for name, m in char_freq.items():
    Gnx.add_node(name, mentions=int(m))
for _, r in cooc.iterrows():
    Gnx.add_edge(r["a"], r["b"], weight=int(r["shared"]))
pos = nx.spring_layout(Gnx, k=0.6, seed=42, weight="weight")

edge_x, edge_y = [], []
for a, b in Gnx.edges():
    edge_x += [pos[a][0], pos[b][0], None]
    edge_y += [pos[a][1], pos[b][1], None]
edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines", hoverinfo="none",
                        line=dict(width=0.6, color="#cccccc"))

nodes = list(Gnx.nodes())
deg_cent = char_pr.reindex(nodes).fillna(char_pr.min()).values
sizes_px = 12 + 60 * (deg_cent - deg_cent.min()) / (np.ptp(deg_cent) or 1)
node_trace = go.Scatter(
    x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes],
    mode="markers+text", text=nodes, textposition="top center",
    textfont=dict(size=9),
    marker=dict(size=sizes_px, color=deg_cent, colorscale="Sunset",
                line=dict(width=1, color="#ffffff"), showscale=True,
                colorbar=dict(title="PageRank")),
    hovertext=[f"{n}: {Gnx.nodes[n]['mentions']} verses" for n in nodes],
    hoverinfo="text",
)
fig4 = go.Figure([edge_trace, node_trace])
fig4.update_layout(
    title="The cast of the Gita: co-occurrence network",
    showlegend=False, xaxis=dict(visible=False), yaxis=dict(visible=False),
    height=640, margin=dict(l=10, r=10, t=40, b=10),
)
assert Gnx.number_of_nodes() == char_freq.shape[0]
export(fig4, "analysis_character_network.html")
fig4

wrote gita-knowledge-graph/exports/analysis_character_network.html


## 6. Narrative arc: theme share across the reading order

Not a graph algorithm: this walks the 700 verses in reading order and tracks
the share of each theme in a rolling window. It shows how the text's centre of
attention shifts, from the opening despair to the turn to knowledge and action,
then the late swell of devotion.

In [19]:
vt = cypher(
    "MATCH (v:Verse)-[m:MENTIONS_THEME]->(t:Theme) "
    "RETURN v.chapter AS chapter, v.verse AS verse, t.name AS theme, m.weight AS weight"
)
order = verse_lookup.reset_index()[["chapter", "verse", "reading_order"]]
vt = vt.merge(order, on=["chapter", "verse"], how="left")
assert vt["reading_order"].notna().all()

WINDOW = 40
max_order = int(order["reading_order"].max())
vt["bin"] = (vt["reading_order"] // WINDOW).astype(int)
grp = vt.groupby(["bin", "theme"], as_index=False)["weight"].sum()
grp["share"] = grp["weight"] / grp.groupby("bin")["weight"].transform("sum")
share = grp
# Each window's shares must sum to 1.
sums = share.groupby("bin")["share"].sum()
assert np.allclose(sums.values, 1.0), sums.to_dict()

focus = ["karma", "jnana", "bhakti", "dharma", "atman", "yoga",
         "detachment", "samsara"]
plot_df = share[share["theme"].isin(focus)].copy()
fig5 = px.area(
    plot_df, x="bin", y="share", color="theme",
    color_discrete_sequence=PALETTE,
    labels=dict(bin=f"reading order (windows of {WINDOW} verses)", share="theme share"),
    title="The arc of attention: theme share across the 700 verses",
)
fig5.update_layout(height=520)
export(fig5, "analysis_narrative_arc.html")
fig5

wrote gita-knowledge-graph/exports/analysis_narrative_arc.html


## 7. Dialogue dynamics: who speaks, to whom

Also descriptive: the Gita is a conversation. This counts verses by speaker per
chapter, and the overall speaker→addressee flow.

In [20]:
spk = cypher(
    "MATCH (v:Verse)-[:SPOKEN_BY]->(p:Person) "
    "RETURN v.chapter AS chapter, p.name AS speaker, count(*) AS verses"
)
assert int(spk["verses"].sum()) == counts["verses"], int(spk["verses"].sum())
fig6a = px.bar(
    spk, x="chapter", y="verses", color="speaker",
    color_discrete_sequence=PALETTE,
    title="Who speaks each chapter", labels=dict(verses="verses"),
)
fig6a.update_layout(height=460, barmode="stack", xaxis=dict(dtick=1))
export(fig6a, "analysis_speakers_by_chapter.html")
fig6a

wrote gita-knowledge-graph/exports/analysis_speakers_by_chapter.html


In [21]:
# Speaker -> addressee flow (Sankey).
flow = cypher(
    "MATCH (p:Person)<-[:SPOKEN_BY]-(v:Verse)-[:ADDRESSED_TO]->(q:Person) "
    "RETURN p.name AS speaker, q.name AS addressee, count(*) AS verses"
)
labels = pd.unique(flow[["speaker", "addressee"]].values.ravel()).tolist()
idx = {name: i for i, name in enumerate(labels)}
fig6b = go.Figure(go.Sankey(
    node=dict(label=labels, pad=18, thickness=16,
              color=[PALETTE[i % len(PALETTE)] for i in range(len(labels))]),
    link=dict(
        source=flow["speaker"].map(idx), target=flow["addressee"].map(idx),
        value=flow["verses"],
    ),
))
fig6b.update_layout(title="Speaker → addressee flow across the whole text",
                    height=420)
assert int(flow["verses"].sum()) == counts["verses"], int(flow["verses"].sum())
export(fig6b, "analysis_dialogue_sankey.html")
fig6b

wrote gita-knowledge-graph/exports/analysis_dialogue_sankey.html


## 8. Cleanup

In [22]:
for name in [SIM_GRAPH, "verse_theme", "verse_concept", "char_cooc"]:
    drop_if_exists(name)
gds.close()
print("dropped projections, closed GDS session.")
print("figures written to", EXPORTS.relative_to(ROOT))

dropped projections, closed GDS session.
figures written to gita-knowledge-graph/exports
